# Workflow 2: end-to-end CNN training (publication-quality rerun)
## BRISC 2025 - brain tumour classification

**Catarina Bota | UCP**

Trains the W2 CNN on the **forensically cleaned** dataset (`brisc2025_clean/`, 5,032 imgs), producing the canonical `modelo_brisc.h5` artefact that downstream notebooks (W3 hybrid, Grad-CAM) reuse.

Improvements vs the original training notebook:
- Trains on the audited dataset (no train/test leakage)
- Stratified train/val split from the training partition (no test set contamination)
- Deterministic seeding (TF + NumPy + Python random) for reproducibility
- Supports **multi-seed** runs (mean and std reported across seeds) for journal-quality variance estimates
- Measures real inference latency with `timeit` (no fabricated numbers)
- Honest reporting: per-class precision/recall/F1, 95% bootstrap CI on accuracy, and confusion matrix CSV

**Prerequisites:**
- Drive mounted with `brisc2025_clean/classification_task/` (output of `forensic_audit_colab.ipynb`)
- GPU runtime (T4 or better)

**Runtime:** ~20-30 min per seed (4 seeds ≈ 1.5-2 h).

## 1. Setup and reproducibility

In [1]:
import os, json, pickle, time, timeit, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, roc_auc_score, classification_report,
                              confusion_matrix, ConfusionMatrixDisplay,
                              precision_recall_fscore_support)
from sklearn.utils import resample

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs available:', gpus)
if not gpus:
    print('WARNING: no GPU detected. Training will be slow.')

2026-07-01 11:29:30.194067: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-01 11:29:30.211128: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1782901770.229284    2782 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1782901770.234883    2782 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1782901770.251184    2782 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

TensorFlow: 2.19.0
GPUs available: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


## 2. Configuration

Set `SEEDS = [42]` for a single deterministic run (~25 min) or `SEEDS = [42, 123, 7, 2025]` for multi-seed variance estimates (~1.5-2 h).

In [2]:
# EXPERIMENT A: unaudited data (BRISC 2025 official split)
DATA_BASE  = '/root/brisc/data_unaudited/classification_task'
TRAIN_DIR  = f'{DATA_BASE}/train'
TEST_DIR   = f'{DATA_BASE}/test'
OUTPUT_DIR = '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/workflow2_resultados_UNAUDITED'
MODEL_OUT  = '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/modelo_brisc_UNAUDITED.h5'

IMG_SIZE   = 128
BATCH_SIZE = 32
MAX_EPOCHS = 60
VAL_SPLIT  = 0.20
CLASSES    = ['glioma', 'meningioma', 'no_tumor', 'pituitary']
N_CLASSES  = len(CLASSES)

SEEDS = [42]  # mesmo seed do audited!

for p in (TRAIN_DIR, TEST_DIR):
    assert os.path.exists(p), f'NOT FOUND: {p}'
print('Config UNAUDITED. Seeds:', SEEDS)

Config UNAUDITED. Seeds: [42]


In [3]:
def set_global_seed(seed):
    """Best-effort deterministic seeding across NumPy, Python random and TensorFlow."""
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed); np.random.seed(seed); tf.random.set_seed(seed)
    # TF deterministic ops (slower but reproducible). Comment out if too slow.
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

## 3. Build the CNN (same architecture as the original W2)

Three Conv2D blocks (32 -> 64 -> 128 filters), each followed by MaxPooling.
Classification head: Flatten -> Dense(128, ReLU) -> Dropout -> Dense(4, softmax).

The Dense(128) layer is the feature layer reused by W3.

In [4]:
def build_cnn(seed):
    set_global_seed(seed)
    init = tf.keras.initializers.GlorotUniform(seed=seed)
    m = models.Sequential([
        layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
        layers.Conv2D(32, 3, padding='same', activation='relu', kernel_initializer=init),
        layers.MaxPooling2D(2),
        layers.Conv2D(64, 3, padding='same', activation='relu', kernel_initializer=init),
        layers.MaxPooling2D(2),
        layers.Conv2D(128, 3, padding='same', activation='relu', kernel_initializer=init),
        layers.MaxPooling2D(2),
        layers.Flatten(),
        layers.Dense(128, activation='relu', kernel_initializer=init, name='feat_dense'),
        layers.Dropout(0.3, seed=seed),
        layers.Dense(N_CLASSES, activation='softmax', kernel_initializer=init, name='predictions'),
    ])
    m.compile(optimizer=optimizers.Adam(learning_rate=1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
    return m

_demo = build_cnn(42)
_demo.summary()

I0000 00:00:1782901786.557596    2782 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5560 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 128, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 64, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 32, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 32768)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ feat_dense (Dense)              │ (None, 128)            │     4,194,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ predictions (Dense)             │ (None, 4)              │           516 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,288,196 (16.36 MB)

 Trainable params: 4,288,196 (16.36 MB)

 Non-trainable params: 0 (0.00 B)

## 4. Data generators

Train: augmentation (rotation 10, zoom 0.1, shift 5%, h-flip) + stratified validation split (20% from train partition).
Test: no augmentation, fixed order (`shuffle=False`) so `classes_` is meaningful.

In [5]:
def make_generators(seed):
    train_aug = ImageDataGenerator(
        rescale=1/255.0,
        rotation_range=10, zoom_range=0.1,
        width_shift_range=0.05, height_shift_range=0.05,
        horizontal_flip=True, validation_split=VAL_SPLIT)
    test_pre = ImageDataGenerator(rescale=1/255.0)
    
    train_gen = train_aug.flow_from_directory(
        TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        classes=CLASSES, class_mode='categorical',
        subset='training', shuffle=True, seed=seed)
    val_gen = train_aug.flow_from_directory(
        TRAIN_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        classes=CLASSES, class_mode='categorical',
        subset='validation', shuffle=False, seed=seed)
    test_gen = test_pre.flow_from_directory(
        TEST_DIR, target_size=(IMG_SIZE, IMG_SIZE), batch_size=BATCH_SIZE,
        classes=CLASSES, class_mode='categorical', shuffle=False)
    return train_gen, val_gen, test_gen

## 5. Training loop (per seed)

In [6]:
def evaluate(model, gen, classes):
    gen.reset()
    y_true = gen.classes
    y_proba = model.predict(gen, verbose=0)
    y_pred = np.argmax(y_proba, axis=1)
    return {
        'accuracy':  accuracy_score(y_true, y_pred) * 100,
        'precision': precision_score(y_true, y_pred, average='macro') * 100,
        'recall':    recall_score(y_true, y_pred, average='macro') * 100,
        'f1':        f1_score(y_true, y_pred, average='macro') * 100,
        'auc':       roc_auc_score(y_true, y_proba, multi_class='ovr', average='macro') * 100,
        'y_true':    y_true, 'y_pred': y_pred, 'y_proba': y_proba,
    }

def bootstrap_acc_ci(y_true, y_pred, n_iter=1000, alpha=0.05, seed=42):
    """Stratified bootstrap CI on accuracy."""
    rng = np.random.RandomState(seed)
    accs = []
    n = len(y_true)
    for _ in range(n_iter):
        idx = rng.randint(0, n, n)
        accs.append((y_true[idx] == y_pred[idx]).mean() * 100)
    lo = np.percentile(accs, 100*alpha/2)
    hi = np.percentile(accs, 100*(1-alpha/2))
    return float(lo), float(hi)

def measure_inference_latency_ms(model, n_warmup=10, n_iter=100):
    """Single-image inference latency in milliseconds."""
    dummy = np.random.rand(1, IMG_SIZE, IMG_SIZE, 3).astype(np.float32)
    for _ in range(n_warmup):
        model(dummy, training=False)
    t = timeit.timeit(lambda: model(dummy, training=False), number=n_iter)
    return (t / n_iter) * 1000.0

In [7]:
all_runs = []  # one dict per seed

for seed in SEEDS:
    print('=' * 60); print(f'  SEED {seed}'); print('=' * 60)
    set_global_seed(seed)
    train_gen, val_gen, test_gen = make_generators(seed)
    print(f'  train batches: {len(train_gen)}  '
          f'val batches: {len(val_gen)}  '
          f'test batches: {len(test_gen)}')
    
    model = build_cnn(seed)
    cbs = [
        callbacks.EarlyStopping(monitor='val_loss', patience=8,
                                 restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5,
                                     patience=4, min_lr=1e-6, verbose=1),
    ]
    t0 = time.time()
    hist = model.fit(train_gen, validation_data=val_gen,
                      epochs=MAX_EPOCHS, callbacks=cbs, verbose=2)
    train_time = time.time() - t0
    
    # Final metrics on the test partition
    test_metrics = evaluate(model, test_gen, CLASSES)
    ci_lo, ci_hi = bootstrap_acc_ci(test_metrics['y_true'],
                                      test_metrics['y_pred'],
                                      n_iter=1000, seed=seed)
    latency_ms = measure_inference_latency_ms(model)
    
    print(f'\n  Test  Accuracy : {test_metrics["accuracy"]:.2f}%  CI95 [{ci_lo:.2f}, {ci_hi:.2f}]')
    print(f'  Test  F1 macro : {test_metrics["f1"]:.2f}%')
    print(f'  Test  AUC      : {test_metrics["auc"]:.2f}%')
    print(f'  Train time     : {train_time:.1f}s  ({len(hist.history["loss"])} epochs)')
    print(f'  Inference      : {latency_ms:.2f} ms/image')
    
    # Save the model for the FIRST seed (consumed by W3 and Grad-CAM)
    if seed == SEEDS[0]:
        model.save(MODEL_OUT)
        print(f'  Model saved to {MODEL_OUT}')
    
    all_runs.append({
        'seed': seed,
        'epochs_trained': len(hist.history['loss']),
        'train_time_s': train_time,
        'inference_ms': latency_ms,
        'test_accuracy': test_metrics['accuracy'],
        'test_precision': test_metrics['precision'],
        'test_recall': test_metrics['recall'],
        'test_f1': test_metrics['f1'],
        'test_auc': test_metrics['auc'],
        'acc_ci_lo': ci_lo, 'acc_ci_hi': ci_hi,
        'history': hist.history,
        'y_true': test_metrics['y_true'].tolist(),
        'y_pred': test_metrics['y_pred'].tolist(),
    })

  SEED 42
Found 4002 images belonging to 4 classes.
Found 998 images belonging to 4 classes.
Found 1000 images belonging to 4 classes.
  train batches: 126  val batches: 32  test batches: 32
Epoch 1/60


I0000 00:00:1782901799.541213    2874 cuda_dnn.cc:529] Loaded cuDNN version 90300


126/126 - 18s - 142ms/step - accuracy: 0.6184 - loss: 0.9127 - val_accuracy: 0.5641 - val_loss: 1.2331 - learning_rate: 0.0010
Epoch 2/60
126/126 - 17s - 131ms/step - accuracy: 0.7659 - loss: 0.5855 - val_accuracy: 0.7685 - val_loss: 0.7636 - learning_rate: 0.0010
Epoch 3/60
126/126 - 16s - 130ms/step - accuracy: 0.8038 - loss: 0.4852 - val_accuracy: 0.7776 - val_loss: 0.7670 - learning_rate: 0.0010
Epoch 4/60
126/126 - 17s - 135ms/step - accuracy: 0.8181 - loss: 0.4429 - val_accuracy: 0.7595 - val_loss: 0.8837 - learning_rate: 0.0010
Epoch 5/60
126/126 - 17s - 131ms/step - accuracy: 0.8428 - loss: 0.3845 - val_accuracy: 0.7816 - val_loss: 0.8972 - learning_rate: 0.0010
Epoch 6/60

Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
126/126 - 16s - 131ms/step - accuracy: 0.8463 - loss: 0.3777 - val_accuracy: 0.7846 - val_loss: 0.9224 - learning_rate: 0.0010
Epoch 7/60
126/126 - 16s - 131ms/step - accuracy: 0.8728 - loss: 0.3298 - val_accuracy: 0.8166 - val_loss:


  Test  Accuracy : 71.60%  CI95 [68.90, 74.30]
  Test  F1 macro : 71.69%
  Test  AUC      : 89.97%
  Train time     : 167.0s  (10 epochs)
  Inference      : 7.21 ms/image
  Model saved to /mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/modelo_brisc_UNAUDITED.h5


## 6. Aggregate across seeds (if multi-seed)

In [13]:
summary = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ('history', 'y_true', 'y_pred')}
    for r in all_runs
])
print(summary.to_string(index=False))
summary.to_csv(f'{OUTPUT_DIR}/w2_per_seed.csv', index=False)

if len(all_runs) > 1:
    agg = pd.DataFrame({
        'metric':   ['accuracy', 'precision', 'recall', 'f1', 'auc'],
        'mean (%)': [np.mean([r[f'test_{m}'] for r in all_runs])
                     for m in ['accuracy','precision','recall','f1','auc']],
        'std (%)':  [np.std([r[f'test_{m}'] for r in all_runs], ddof=1)
                     for m in ['accuracy','precision','recall','f1','auc']],
    })
    print('\nAggregated across', len(all_runs), 'seeds:')
    print(agg.round(2).to_string(index=False))
    agg.to_csv(f'{OUTPUT_DIR}/w2_aggregated.csv', index=False)

 seed  epochs_trained  train_time_s  inference_ms  test_accuracy  test_precision  test_recall   test_f1  test_auc  acc_ci_lo  acc_ci_hi
   42              23    336.688887      4.687697       88.20059       88.382926    88.781338 88.339683 97.670999  85.545723  90.560472


OSError: Cannot save file into a non-existent directory: '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/workflow2_resultados'

## 7. Confusion matrix and per-class metrics (first seed)

In [12]:
r0 = all_runs[0]
y_true = np.array(r0['y_true']); y_pred = np.array(r0['y_pred'])
cm = confusion_matrix(y_true, y_pred)

pd.DataFrame(cm, index=CLASSES, columns=CLASSES).to_csv(
    f'{OUTPUT_DIR}/w2_confusion_matrix.csv')

prec_c, rec_c, f1_c, sup_c = precision_recall_fscore_support(
    y_true, y_pred, labels=range(N_CLASSES))
perclass = pd.DataFrame({
    'class':     CLASSES,
    'precision': (prec_c*100).round(2),
    'recall':    (rec_c*100).round(2),
    'f1':        (f1_c*100).round(2),
    'support':   sup_c.astype(int),
})
print(perclass.to_string(index=False))
perclass.to_csv(f'{OUTPUT_DIR}/w2_perclass.csv', index=False)

fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay(cm, display_labels=CLASSES).plot(
    cmap='Greens', ax=ax, values_format='d', colorbar=True)
ax.set_title(f'W2 CNN - test accuracy {r0["test_accuracy"]:.2f}%  '
             f'(95% CI [{r0["acc_ci_lo"]:.1f}, {r0["acc_ci_hi"]:.1f}])',
             fontsize=11, fontweight='bold')
plt.savefig(f'{OUTPUT_DIR}/w2_confusion_matrix.png', bbox_inches='tight', dpi=150)
plt.savefig(f'{OUTPUT_DIR}/w2_confusion_matrix.pdf', bbox_inches='tight')
plt.show()

OSError: Cannot save file into a non-existent directory: '/mnt/c/Users/cbot/Desktop/BRISC pos graduação/brisc_gui/workflow2_resultados'

## 8. Training curves (first seed)

In [ ]:
h = r0['history']
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(h['loss'], label='train'); axes[0].plot(h['val_loss'], label='val')
axes[0].set_xlabel('epoch'); axes[0].set_ylabel('loss')
axes[0].set_title('Loss'); axes[0].legend()
axes[1].plot(h['accuracy'], label='train'); axes[1].plot(h['val_accuracy'], label='val')
axes[1].set_xlabel('epoch'); axes[1].set_ylabel('accuracy')
axes[1].set_title('Accuracy'); axes[1].legend()
for ax in axes:
    for s in ['top','right']: ax.spines[s].set_visible(False)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}/w2_training_history.png', bbox_inches='tight', dpi=150)
plt.show()

# Save the raw history JSON for the article (figure regeneration)
with open(f'{OUTPUT_DIR}/w2_history.json', 'w') as f:
    json.dump({k: [float(x) for x in v] for k, v in h.items()}, f, indent=2)

## 9. Saved artifacts

Files generated in `workflow2_resultados/`:

In [ ]:
for f in sorted(os.listdir(OUTPUT_DIR)):
    sz = os.path.getsize(f'{OUTPUT_DIR}/{f}') / 1024
    print(f'  {f}  ({sz:.1f} KB)')
print(f'\nCNN model: {MODEL_OUT}')

## Next steps

1. **W3 (hybrid)**: open `workflow3_hybrid_all_classifiers.ipynb` and delete `workflow3_resultados/cnn_features.npz` from Drive. Re-running W3 will extract features from this freshly trained W2 model on the cleaned dataset (no cache contamination).
2. **Grad-CAM**: rerun `workflow3_gradcam.ipynb` — it auto-detects the last Conv2D layer and works without changes.
3. **Multi-seed for journal**: set `SEEDS = [42, 123, 7, 2025]` and re-run. Reported means and standard deviations replace the previous single-seed point estimates.

## What this notebook fixes vs the original W2

- Trains on `brisc2025_clean/` (5,032 imgs, no MD5/pHash duplicates between train and test)
- Uses fixed seeds across NumPy, Python `random`, and TensorFlow for reproducibility
- Measures inference latency with `timeit` (replaces fabricated values)
- Reports a real bootstrap 95% CI on test accuracy (replaces fabricated values)
- Saves training history JSON for figure regeneration
- Supports multi-seed averaging for proper variance estimates in the journal version